# Data Processing
## Personal Cognitive Load & Focus Monitoring System

This notebook processes raw data and prepares it for model training.

### Objectives:
1. Load raw data
2. Data cleaning and validation
3. Feature engineering
4. Data transformation
5. Save processed data


In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add src to path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / "src"))

# Set paths
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {DATA_RAW}")
print(f"Processed data: {DATA_PROCESSED}")


Project root: /home/pnmynh/Documents/personal-cognitive-load-monitor
Raw data: /home/pnmynh/Documents/personal-cognitive-load-monitor/data/raw
Processed data: /home/pnmynh/Documents/personal-cognitive-load-monitor/data/processed


In [2]:
# Load raw data
raw_file = DATA_RAW / "sample_data.csv"
if not raw_file.exists():
    from data_generator import generate_sample_data
    generate_sample_data(n_samples=200, output_path=raw_file)

df = pd.read_csv(raw_file)
print(f"Loaded {len(df)} records")
df.head()


Loaded 200 records


,date,focus_minutes,distraction_minutes,tasks_due,hours_to_deadline,cognitive_load_level
0,2026-01-13,185,133,5,16.89,HIGH
1,2025-12-28,285,38,5,67.84,MEDIUM
2,2026-01-04,178,292,0,155.32,LOW
3,2025-12-20,312,72,3,97.60,LOW
4,2025-12-23,226,82,1,162.45,LOW


In [3]:
# Data cleaning
print("Before cleaning:")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

# Remove duplicates
df = df.drop_duplicates()
print(f"\nAfter removing duplicates: {df.shape}")

# Validate data ranges
df = df[
    (df["focus_minutes"] >= 0) &
    (df["distraction_minutes"] >= 0) &
    (df["tasks_due"] >= 0) &
    (df["hours_to_deadline"] >= 0)
]
print(f"After validation: {df.shape}")

# Check for any remaining issues
print(f"\nFinal missing values: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")


Before cleaning:
Shape: (200, 6)
Missing values: 0

After removing duplicates: (200, 6)
After validation: (200, 6)

Final missing values: 0
Final shape: (200, 6)


In [4]:
# Feature engineering
df_processed = df.copy()

# Calculate derived features
df_processed["total_minutes"] = df_processed["focus_minutes"] + df_processed["distraction_minutes"]
df_processed["focus_ratio"] = df_processed["focus_minutes"] / df_processed["total_minutes"].replace(0, 1)
df_processed["distraction_ratio"] = df_processed["distraction_minutes"] / df_processed["total_minutes"].replace(0, 1)

# Deadline pressure categories
df_processed["deadline_pressure"] = pd.cut(
    df_processed["hours_to_deadline"],
    bins=[0, 24, 72, float("inf")],
    labels=["HIGH", "MEDIUM", "LOW"]
)

# Task load categories
df_processed["task_load"] = pd.cut(
    df_processed["tasks_due"],
    bins=[0, 2, 5, float("inf")],
    labels=["LOW", "MEDIUM", "HIGH"]
)

print("New features created:")
print(df_processed[["focus_ratio", "distraction_ratio", "deadline_pressure", "task_load"]].head())


New features created:
   focus_ratio  distraction_ratio deadline_pressure task_load
0     0.581761           0.418239              HIGH    MEDIUM
1     0.882353           0.117647            MEDIUM    MEDIUM
2     0.378723           0.621277               LOW       NaN
3     0.812500           0.187500               LOW    MEDIUM
4     0.733766           0.266234               LOW       LOW


In [5]:
# Prepare features for modeling
# Select features for training
feature_columns = [
    "focus_minutes",
    "distraction_minutes",
    "tasks_due",
    "hours_to_deadline",
    "focus_ratio",
    "distraction_ratio",
]

X = df_processed[feature_columns].copy()
y = df_processed["cognitive_load_level"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:", feature_columns)
print("\nTarget distribution:")
print(y.value_counts())


Features shape: (200, 6)
Target shape: (200,)

Feature columns: ['focus_minutes', 'distraction_minutes', 'tasks_due', 'hours_to_deadline', 'focus_ratio', 'distraction_ratio']

Target distribution:
cognitive_load_level
MEDIUM    120
LOW        49
HIGH       31
Name: count, dtype: int64


In [6]:
# Save processed data
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Save full processed dataset
processed_file = DATA_PROCESSED / "processed_data.csv"
df_processed.to_csv(processed_file, index=False)
print(f"Saved processed data to {processed_file}")

# Save features and target separately
X_file = DATA_PROCESSED / "features.csv"
y_file = DATA_PROCESSED / "target.csv"

X.to_csv(X_file, index=False)
y.to_csv(y_file, index=False)

print(f"Saved features to {X_file}")
print(f"Saved target to {y_file}")
print("\nData processing complete!")


Saved processed data to /home/pnmynh/Documents/personal-cognitive-load-monitor/data/processed/processed_data.csv
Saved features to /home/pnmynh/Documents/personal-cognitive-load-monitor/data/processed/features.csv
Saved target to /home/pnmynh/Documents/personal-cognitive-load-monitor/data/processed/target.csv

Data processing complete!
